# 📋 Scenario 3 — Job & Internship Application Tracker
### Ubuntu Automation Assignment · Group Project

---

**Group Scenario:** `Scenario 3 — Job / Internship Application Tracker`  
**Environment:** Google Colab (Ubuntu-based)  
**Dataset:** 50 simulated applications across Korean tech companies, Jan–Mar 2024

---

## 🎯 What This Scenario Automates

Managing a job search across 50+ companies means tracking dozens of dates, statuses, salary figures, and follow-up deadlines — work that is normally done by hand in a spreadsheet. This notebook automates the entire analytical side of that process:

| Part | What It Does |
|------|-------------|
| **Part 1** | Load & clean the dataset, handle missing values, engineer features |
| **Part 2** | Full status dashboard — where every application stands |
| **Part 3** | Follow-up alert system — who needs a nudge today |
| **Part 4** | Recruitment funnel analysis — where candidates drop off |
| **Part 5** | Platform & salary intelligence — where to apply and what to expect |
| **Part 6** | Timeline & activity analysis — how the search evolved over time |
| **Part 7** | Automated weekly report generator — a ready-to-send digest |

> 📌 **Note:** This is one scenario in a group assignment. Each group member implemented a different automation scenario. This notebook covers the job tracker scenario end-to-end.


---
## ⚙️ Setup — Libraries & Configuration

We import all required libraries upfront. All of these come pre-installed in Google Colab — no `!pip install` needed.


In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import matplotlib.gridspec as gridspec
import seaborn as sns
import numpy as np
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Global style settings
plt.rcParams.update({
    'figure.facecolor': '#FAFAFA',
    'axes.facecolor':   '#FAFAFA',
    'axes.spines.top':  False,
    'axes.spines.right':False,
    'font.size': 11,
})
sns.set_palette("husl")

TODAY = pd.Timestamp('2024-04-10')   # simulated "today"
print(f"✅ Libraries loaded. Simulated date: {TODAY.date()}")


---
## Part 1 — Data Loading & Cleaning

### Why this step matters

Raw data always has issues: missing dates, inconsistent text, columns that need to be derived. Before any analysis, we need to load the dataset, inspect it, handle `NaN` values, and engineer useful features like `days_to_response` and `salary_midpoint`.

In a real Ubuntu automation system, this cleaning step would run automatically every time new rows are added to the CSV (triggered by a `inotifywait` file watcher or a cron job).


In [ ]:
# ── Load ──────────────────────────────────────────────────────────────────────
df = pd.read_csv('job_applications.csv',
                 parse_dates=['date_applied', 'follow_up_date', 'response_date'])

print(f"Shape: {df.shape[0]} rows × {df.shape[1]} columns")
print(f"Date range: {df['date_applied'].min().date()} → {df['date_applied'].max().date()}")
df.head(8)


In [ ]:
# ── Inspect missing values ────────────────────────────────────────────────────
missing = df.isnull().sum()
missing = missing[missing > 0]
print("Columns with missing values:")
print(missing.to_string())
print()

# Expected: response_date (ghosted/applied/interview apps have no response yet)
# follow_up_date, rejection_stage, notes are also sometimes empty — that's fine


In [ ]:
# ── Feature engineering ──────────────────────────────────────────────────────

# Days from application to first response (only for apps that got a response)
df['days_to_response'] = (df['response_date'] - df['date_applied']).dt.days

# Salary midpoint for fair comparisons
df['salary_mid'] = (df['salary_min'] + df['salary_max']) / 2

# Month and week applied
df['month_applied'] = df['date_applied'].dt.to_period('M').astype(str)
df['week_applied']  = df['date_applied'].dt.to_period('W').apply(lambda r: r.start_time)

# Days since application (from simulated today)
df['days_since_applied'] = (TODAY - df['date_applied']).dt.days

# Did this app get any response at all?
df['got_response'] = df['response_date'].notna().astype(int)

# Did it reach at least the interview stage?
df['reached_interview'] = df['status'].isin(['Interview', 'Offer']).astype(int)

print("✅ Feature engineering complete. New columns:")
new_cols = ['days_to_response','salary_mid','month_applied','week_applied',
            'days_since_applied','got_response','reached_interview']
print(df[new_cols].head(5).to_string())


In [ ]:
# ── Summary statistics ────────────────────────────────────────────────────────
print("=== Dataset Summary ===")
print(f"Total applications     : {len(df)}")
print(f"Unique companies       : {df['company'].nunique()}")
print(f"Internships            : {(df['type']=='Internship').sum()}")
print(f"Full-time roles        : {(df['type']=='Full-time').sum()}")
print(f"Remote positions       : {(df['remote']=='Yes').sum()}")
print(f"Used referral          : {(df['referral']=='Yes').sum()}")
print(f"Submitted cover letter : {(df['cover_letter']=='Yes').sum()}")
print()
print("=== Status Breakdown ===")
print(df['status'].value_counts().to_string())
print()
print("=== Avg days to response (when responded) ===")
print(f"{df['days_to_response'].mean():.1f} days")


---
## Part 2 — Status Dashboard

### What this visualises

A full overview of where all 50 applications stand. This is the "homepage" of the tracker — the first thing you'd want to see when you open your job search tool in the morning.

We generate:
1. A **donut chart** showing the pipeline composition
2. A **grouped bar chart** breaking down status by job type (Internship vs Full-time)
3. Key **KPI metrics** printed as a summary


In [ ]:
STATUS_COLORS = {
    'Applied'  : '#4A90D9',
    'Interview': '#F5A623',
    'Offer'    : '#27AE60',
    'Rejected' : '#E74C3C',
    'Ghosted'  : '#95A5A6',
}

status_counts = df['status'].value_counts()

# ── KPIs ──────────────────────────────────────────────────────────────────────
total      = len(df)
offers     = (df['status'] == 'Offer').sum()
interviews = df['reached_interview'].sum()
ghosted    = (df['status'] == 'Ghosted').sum()
rejected   = (df['status'] == 'Rejected').sum()

print("┌─────────────────────────────────────────────┐")
print("│         APPLICATION PIPELINE SUMMARY        │")
print("├─────────────────────────────────────────────┤")
print(f"│  Total Applications       {total:>5}             │")
print(f"│  Offers Received          {offers:>5}  ({offers/total*100:.1f}%)    │")
print(f"│  Reached Interview        {interviews:>5}  ({interviews/total*100:.1f}%)    │")
print(f"│  Rejected                 {rejected:>5}  ({rejected/total*100:.1f}%)    │")
print(f"│  Ghosted (no response)    {ghosted:>5}  ({ghosted/total*100:.1f}%)     │")
print("└─────────────────────────────────────────────┘")


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# Left — donut chart
colors  = [STATUS_COLORS.get(s, '#BDC3C7') for s in status_counts.index]
wedges, texts, autotexts = axes[0].pie(
    status_counts.values,
    labels      = status_counts.index,
    colors      = colors,
    autopct     = '%1.1f%%',
    startangle  = 140,
    pctdistance = 0.78,
    wedgeprops  = dict(width=0.52, edgecolor='white', linewidth=2.5)
)
for t in autotexts: t.set_fontsize(10); t.set_fontweight('bold')
axes[0].text(0, 0, f'{total}\nApps', ha='center', va='center',
             fontsize=17, fontweight='bold', color='#2C3E50')
axes[0].set_title('Overall Pipeline Status', fontsize=13, fontweight='bold', pad=15)

# Right — grouped bar by type
type_status = df.groupby(['type','status']).size().unstack(fill_value=0)
type_status = type_status.reindex(columns=STATUS_COLORS.keys(), fill_value=0)
x = np.arange(len(type_status))
width = 0.15
for i, (col, color) in enumerate(STATUS_COLORS.items()):
    if col in type_status.columns:
        axes[1].bar(x + i*width, type_status[col], width,
                    label=col, color=color, edgecolor='white', linewidth=1.2)
axes[1].set_xticks(x + width * 2)
axes[1].set_xticklabels(type_status.index, fontsize=12)
axes[1].set_ylabel('Applications')
axes[1].set_title('Status by Job Type', fontsize=13, fontweight='bold')
axes[1].legend(loc='upper right', frameon=True, fontsize=9)

fig.suptitle('Job Application Dashboard', fontsize=16, fontweight='bold', y=1.01)
plt.tight_layout()
plt.savefig('dashboard.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Dashboard saved.")


---
## Part 3 — Automated Follow-up Alert System

### What this simulates

In a real Ubuntu setup, this logic would live inside a Python script called by `cron` every morning at 08:00. It checks each active application, computes whether a follow-up is overdue, due today, or upcoming, and prints an alert list — like a to-do email digest.

Every job seeker knows the problem: you applied two weeks ago, heard nothing, and forgot to follow up. This system prevents that.

**Alert levels:**

| Level | Meaning |
|-------|---------|
| 🔴 OVERDUE | Follow-up date passed, status still `Applied` |
| 🟡 DUE TODAY | Follow-up is due on today's simulated date |
| 🟢 UPCOMING | Due within the next 7 days |
| ⚪ FUTURE | Scheduled but not urgent yet |


In [ ]:
# Only check apps that are still active (Applied or Interview)
active = df[df['status'].isin(['Applied', 'Interview'])].copy()
active['days_to_followup'] = (active['follow_up_date'] - TODAY).dt.days

def alert_level(days):
    if pd.isna(days):   return '⚫ No Date Set'
    if days < 0:        return '🔴 OVERDUE'
    if days == 0:       return '🟡 DUE TODAY'
    if days <= 7:       return '🟢 UPCOMING'
    return '⚪ Future'

active['alert'] = active['days_to_followup'].apply(alert_level)

# Print alert digest
print(f"🤖 Follow-up Alert Digest — {TODAY.date()}")
print("=" * 62)
for level in ['🔴 OVERDUE', '🟡 DUE TODAY', '🟢 UPCOMING']:
    subset = active[active['alert'] == level][
        ['company','role','type','date_applied','days_since_applied','days_to_followup']
    ].sort_values('days_to_followup')
    if not subset.empty:
        print(f"\n  {level}  —  {len(subset)} application(s)")
        print(subset.to_string(index=False))
print("\n" + "=" * 62)
print(f"  Total active applications checked: {len(active)}")


In [ ]:
# ── Visualise alert distribution ─────────────────────────────────────────────
alert_counts = active['alert'].value_counts()
alert_colors_map = {
    '🔴 OVERDUE'    : '#E74C3C',
    '🟡 DUE TODAY'  : '#F39C12',
    '🟢 UPCOMING'   : '#27AE60',
    '⚪ Future'      : '#BDC3C7',
    '⚫ No Date Set' : '#7F8C8D',
}

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Left — alert bar chart
bars = axes[0].barh(
    alert_counts.index, alert_counts.values,
    color=[alert_colors_map.get(k,'#aaa') for k in alert_counts.index],
    edgecolor='white', linewidth=1.5, height=0.55
)
for bar in bars:
    axes[0].text(bar.get_width() + 0.15, bar.get_y() + bar.get_height()/2,
                 str(int(bar.get_width())), va='center', fontweight='bold')
axes[0].set_xlabel('Number of Applications')
axes[0].set_title('Follow-up Alert Levels', fontweight='bold', fontsize=13)
axes[0].set_xlim(0, alert_counts.max() + 3)

# Right — days since applied for active apps (sorted)
active_sorted = active.sort_values('days_since_applied', ascending=True)
bar_colors = ['#E74C3C' if d > 21 else '#F39C12' if d > 14 else '#27AE60'
              for d in active_sorted['days_since_applied']]
axes[1].barh(active_sorted['company'] + ' (' + active_sorted['role'].str[:15] + ')',
             active_sorted['days_since_applied'],
             color=bar_colors, edgecolor='white', linewidth=1)
axes[1].axvline(14, color='#F39C12', linestyle='--', linewidth=1.5, label='14 days')
axes[1].axvline(21, color='#E74C3C', linestyle='--', linewidth=1.5, label='21 days')
axes[1].set_xlabel('Days Since Applied')
axes[1].set_title('Days Since Application (Active Only)', fontweight='bold', fontsize=13)
axes[1].legend(fontsize=9)

fig.suptitle('Follow-up Alert System', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('followup_alerts.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Alert chart saved.")


---
## Part 4 — Recruitment Funnel Analysis

### What this shows

Most job seekers don't know their personal conversion rates. This part automatically calculates:
- What % of applications made it to the interview stage
- What % converted to an offer
- Where rejections happened (HR round vs Technical round vs Final round)
- How Internship and Full-time tracks compare

This is the same kind of funnel analysis used in sales pipelines — applied to a job search.


In [ ]:
# ── Funnel conversion rates ───────────────────────────────────────────────────
def funnel(data, label="Overall"):
    n = len(data)
    n_inter = data['reached_interview'].sum()
    n_offer = (data['status'] == 'Offer').sum()
    print(f"  [{label}]  Applied: {n}  →  Interview: {n_inter} ({n_inter/n*100:.0f}%)  →  Offer: {n_offer} ({n_offer/n*100:.0f}%)")
    return {'label': label, 'Applied': n, 'Interview': n_inter, 'Offer': n_offer}

print("Funnel Conversion Summary")
print("=" * 65)
rows = []
rows.append(funnel(df, "Overall"))
rows.append(funnel(df[df['type']=='Internship'], "Internship"))
rows.append(funnel(df[df['type']=='Full-time'],  "Full-time"))
rows.append(funnel(df[df['referral']=='Yes'],    "With Referral"))
rows.append(funnel(df[df['referral']=='No'],     "No Referral"))
rows.append(funnel(df[df['cover_letter']=='Yes'],"Cover Letter"))
rows.append(funnel(df[df['cover_letter']=='No'], "No Cover Letter"))


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 5))

funnel_groups = [
    ("Overall", df,                           '#3498DB'),
    ("Internship", df[df['type']=='Internship'], '#9B59B6'),
    ("Full-time",  df[df['type']=='Full-time'],  '#E67E22'),
]

for ax, (title, data, color) in zip(axes, funnel_groups):
    stages = ['Applied', 'Interview', 'Offer']
    vals = [
        len(data),
        data['reached_interview'].sum(),
        (data['status']=='Offer').sum()
    ]
    bars = ax.bar(stages, vals, color=color, alpha=0.82,
                  edgecolor='white', linewidth=2.5, width=0.5)
    for bar, val, total in zip(bars, vals, [vals[0]]*3):
        pct = val/vals[0]*100 if vals[0] > 0 else 0
        ax.text(bar.get_x() + bar.get_width()/2,
                bar.get_height() + 0.4,
                f'{val}\n({pct:.0f}%)',
                ha='center', fontsize=11, fontweight='bold', color='#2C3E50')
    ax.set_ylim(0, vals[0] + 8)
    ax.set_title(title, fontsize=13, fontweight='bold', color=color)
    ax.set_ylabel('Applications')

fig.suptitle('Recruitment Funnel — Conversion by Track', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('funnel.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Rejection stage breakdown ─────────────────────────────────────────────────
rejected = df[df['status'] == 'Rejected'].copy()
stage_counts = rejected['rejection_stage'].fillna('Unknown').value_counts()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

stage_colors = ['#E74C3C','#E67E22','#F39C12','#BDC3C7','#95A5A6']
axes[0].bar(stage_counts.index, stage_counts.values,
            color=stage_colors[:len(stage_counts)], edgecolor='white', linewidth=2)
for i, v in enumerate(stage_counts.values):
    axes[0].text(i, v + 0.15, str(v), ha='center', fontweight='bold')
axes[0].set_title('Rejections by Stage', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].set_xlabel('Stage Rejected At')

# Referral vs no referral outcome comparison
ref_outcome = df.groupby(['referral','status']).size().unstack(fill_value=0)
ref_outcome.plot(kind='bar', ax=axes[1],
                 color=[STATUS_COLORS.get(c,'#aaa') for c in ref_outcome.columns],
                 edgecolor='white', linewidth=1.5)
axes[1].set_title('Outcomes: Referral vs No Referral', fontweight='bold', fontsize=13)
axes[1].set_xlabel('Had Referral?')
axes[1].set_ylabel('Applications')
axes[1].legend(title='Status', fontsize=9)
axes[1].tick_params(axis='x', rotation=0)

fig.suptitle('Rejection Analysis', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('rejection_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


---
## Part 5 — Platform & Salary Intelligence

### What this automates

Every job seeker wonders: *"Should I spend more time on LinkedIn or Wanted? What salary should I target?"* Without data, this is pure guesswork. This part automatically computes:

- **Response rate per platform** — which platforms actually lead to replies
- **Average salary offered per platform** — where are the higher-paying jobs posted
- **Salary distribution** by job type (internship vs full-time)
- **Resume version performance** — which resume version got more traction


In [ ]:
# ── Platform stats ────────────────────────────────────────────────────────────
df['responded'] = df['status'].isin(['Interview','Offer','Rejected']).astype(int)

platform_stats = df.groupby('platform').agg(
    Applications  = ('application_id', 'count'),
    Avg_Salary_M  = ('salary_mid', lambda x: round(x.mean()/1_000_000, 2)),
    Response_Rate = ('responded', lambda x: round(x.mean()*100, 1)),
    Offer_Rate    = ('status', lambda x: round((x=='Offer').mean()*100, 1)),
    Avg_Interviews= ('interview_rounds', 'mean')
).round(2).reset_index().sort_values('Response_Rate', ascending=False)

print("Platform Intelligence Report")
print("=" * 65)
print(platform_stats.to_string(index=False))


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# 1 — Response rate by platform
ps = platform_stats.sort_values('Response_Rate')
axes[0,0].barh(ps['platform'], ps['Response_Rate'],
               color='#3498DB', edgecolor='white', linewidth=1.5)
for i, v in enumerate(ps['Response_Rate']):
    axes[0,0].text(v+0.5, i, f'{v}%', va='center', fontweight='bold')
axes[0,0].set_title('Response Rate by Platform', fontweight='bold', fontsize=12)
axes[0,0].set_xlabel('Response Rate (%)')

# 2 — Avg salary by platform
ps2 = platform_stats.sort_values('Avg_Salary_M')
axes[0,1].barh(ps2['platform'], ps2['Avg_Salary_M'],
               color='#27AE60', edgecolor='white', linewidth=1.5)
for i, v in enumerate(ps2['Avg_Salary_M']):
    axes[0,1].text(v+0.02, i, f'₩{v:.2f}M', va='center', fontweight='bold')
axes[0,1].set_title('Avg Salary by Platform (₩M/month)', fontweight='bold', fontsize=12)
axes[0,1].set_xlabel('Avg Mid Salary (Millions ₩)')

# 3 — Salary box by job type
data_intern = df[df['type']=='Internship']['salary_mid'] / 1_000_000
data_full   = df[df['type']=='Full-time']['salary_mid'] / 1_000_000
axes[1,0].boxplot([data_intern, data_full], labels=['Internship','Full-time'],
                  patch_artist=True,
                  boxprops=dict(facecolor='#D6EAF8', color='#2C3E50'),
                  medianprops=dict(color='#E74C3C', linewidth=2.5),
                  whiskerprops=dict(color='#7F8C8D'),
                  flierprops=dict(marker='o', markerfacecolor='#E74C3C', markersize=6))
axes[1,0].set_title('Salary Distribution by Job Type', fontweight='bold', fontsize=12)
axes[1,0].set_ylabel('Mid Salary (₩ Millions/month)')

# 4 — Resume version performance
resume_perf = df.groupby('resume_version').agg(
    Applications     = ('application_id','count'),
    Interview_Rate   = ('reached_interview','mean'),
    Response_Rate    = ('responded','mean')
).reset_index()
x = np.arange(len(resume_perf))
w = 0.3
axes[1,1].bar(x - w/2, resume_perf['Interview_Rate']*100, w,
              label='Interview Rate %', color='#F39C12', edgecolor='white')
axes[1,1].bar(x + w/2, resume_perf['Response_Rate']*100, w,
              label='Response Rate %', color='#9B59B6', edgecolor='white')
axes[1,1].set_xticks(x)
axes[1,1].set_xticklabels(resume_perf['resume_version'])
axes[1,1].set_title('Resume Version Performance', fontweight='bold', fontsize=12)
axes[1,1].set_ylabel('Rate (%)')
axes[1,1].legend(fontsize=9)

fig.suptitle('Platform & Salary Intelligence Dashboard', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.savefig('platform_salary.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Intelligence dashboard saved.")


---
## Part 6 — Timeline & Activity Analysis

### What this shows

A job search unfolds over months. This part visualises how application activity changed over time and how quickly companies responded. Key questions answered:

- How many applications were sent each week?
- Did the pace slow down or speed up over time?
- What's the typical turnaround time from apply → response?
- Which companies took the longest to respond?


In [ ]:
# ── Weekly activity ──────────────────────────────────────────────────────────
weekly = df.groupby('week_applied').agg(
    Applied    = ('application_id','count'),
    Interviews = ('reached_interview','sum'),
    Offers     = ('status', lambda x: (x=='Offer').sum()),
    Rejections = ('status', lambda x: (x=='Rejected').sum()),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(13, 9))

# Top — weekly applications timeline
axes[0].fill_between(weekly['week_applied'], weekly['Applied'],
                     alpha=0.12, color='#3498DB')
axes[0].plot(weekly['week_applied'], weekly['Applied'],   'o-',
             color='#3498DB', lw=2.5, ms=7, label='Applied')
axes[0].plot(weekly['week_applied'], weekly['Interviews'],'s-',
             color='#F39C12', lw=2,   ms=7, label='Interviews')
axes[0].plot(weekly['week_applied'], weekly['Offers'],    '^-',
             color='#27AE60', lw=2,   ms=8, label='Offers')
axes[0].plot(weekly['week_applied'], weekly['Rejections'],'v--',
             color='#E74C3C', lw=1.8, ms=6, label='Rejections')
axes[0].set_title('Weekly Application Activity', fontweight='bold', fontsize=13)
axes[0].set_ylabel('Count')
axes[0].legend(loc='upper right', fontsize=10)
axes[0].tick_params(axis='x', rotation=25)

# Bottom — response time distribution
responded = df[df['days_to_response'].notna()].copy()
axes[1].hist(responded['days_to_response'], bins=15,
             color='#9B59B6', edgecolor='white', linewidth=1.5, alpha=0.85)
avg_days = responded['days_to_response'].mean()
axes[1].axvline(avg_days, color='#E74C3C', linestyle='--', lw=2,
                label=f'Avg: {avg_days:.1f} days')
axes[1].set_title('Distribution of Response Times (Days to First Response)',
                  fontweight='bold', fontsize=13)
axes[1].set_xlabel('Days from Application to Response')
axes[1].set_ylabel('Number of Applications')
axes[1].legend(fontsize=10)

plt.tight_layout()
plt.savefig('timeline_analysis.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Monthly summary heatmap ────────────────────────────────────────────────────
monthly_status = df.groupby(['month_applied','status']).size().unstack(fill_value=0)
monthly_status = monthly_status.reindex(columns=list(STATUS_COLORS.keys()), fill_value=0)

fig, ax = plt.subplots(figsize=(10, 4))
sns.heatmap(monthly_status.T, annot=True, fmt='d', cmap='YlOrRd',
            linewidths=0.5, linecolor='white', ax=ax,
            cbar_kws={'label': 'Applications'})
ax.set_title('Applications by Month & Status (Heatmap)', fontweight='bold', fontsize=13)
ax.set_xlabel('Month Applied')
ax.set_ylabel('Status')
plt.tight_layout()
plt.savefig('monthly_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()
print("✅ Timeline charts saved.")


---
## Part 7 — Automated Weekly Report Generator

### What this automates

The final scenario generates a complete **Markdown report** automatically — the kind of summary you'd want emailed to yourself every Monday morning. In a real Ubuntu setup, this script would be the last step in a cron pipeline:

```
0 8 * * 1  python3 /home/user/job_tracker/generate_report.py | mail -s "Weekly Job Report" me@email.com
```

The report includes all key metrics, current action items, and a list of active offers to decide on.


In [ ]:
# ── Build the automated report ───────────────────────────────────────────────
offers_list   = df[df['status'] == 'Offer']
overdue       = active[active['alert'] == '🔴 OVERDUE']
in_interview  = df[df['status'] == 'Interview']
avg_resp      = df['days_to_response'].mean()

report = f"""# 📬 Job Search Weekly Report
**Generated:** {TODAY.strftime('%A, %B %d, %Y')}
**Data as of:** {df['date_applied'].max().date()}

---

## 📊 Overall Progress

| Metric | Value |
|--------|-------|
| Total Applications Sent | {len(df)} |
| Internships | {(df['type']=='Internship').sum()} |
| Full-time Roles | {(df['type']=='Full-time').sum()} |
| Interviews Reached | {df['reached_interview'].sum()} ({df['reached_interview'].mean()*100:.1f}%) |
| Offers Received | {len(offers_list)} ({len(offers_list)/len(df)*100:.1f}%) |
| Ghosted (no reply) | {(df['status']=='Ghosted').sum()} |
| Avg Days to Response | {avg_resp:.1f} days |

---

## 🏆 Active Offers

"""
for _, row in offers_list.iterrows():
    report += f"- ✅ **{row['company']}** — {row['role']} ({row['type']}) · ₩{row['salary_mid']:,.0f}/mo\n"

report += f"""
---

## 🔄 Interviews In Progress ({len(in_interview)})

"""
for _, row in in_interview.iterrows():
    report += f"- 🟡 **{row['company']}** — {row['role']} · Round {row['interview_rounds']} done\n"

report += f"""
---

## ⚠️ Action Items

- {len(overdue)} applications are **OVERDUE** for follow-up
- {len(active)} total active applications need monitoring
- {(df['status']=='Applied').sum()} applications still awaiting any response

"""

if not overdue.empty:
    report += "### Overdue Follow-ups\n"
    for _, row in overdue.iterrows():
        report += f"  - {row['company']} ({row['role']}) — applied {row['days_since_applied']} days ago\n"

report += f"""
---

## 📈 Best Performing Platform This Period
"""
best_platform = platform_stats.loc[platform_stats['Response_Rate'].idxmax()]
report += f"**{best_platform['platform']}** with {best_platform['Response_Rate']}% response rate across {best_platform['Applications']} applications.\n"

report += """
---
*Auto-generated by Job Tracker Automation Script*  
*Scheduled via cron: `0 8 * * 1 python3 generate_report.py`*
"""

with open('weekly_report.md', 'w') as f:
    f.write(report)

print(report)
print("\n✅ Report saved to weekly_report.md")


---
## 💭 Reflection

This scenario taught me that automation is not just about saving time — it's about making **invisible information visible**. Before building this tracker, all 50 applications existed as separate rows in a spreadsheet. After automating the analysis, patterns emerged immediately: which platforms perform best, where in the funnel candidates drop off, how long companies actually take to respond.

The most technically interesting part was the **follow-up alert system** in Part 3. At first glance it seems trivial — just compare two dates. But handling edge cases (ghosted apps with no follow-up date, interview-stage apps with rolling deadlines, apps from different time zones) made it much more complex. This mirrors exactly what real CRM systems do internally.

I also learned to appreciate **feature engineering**. Adding derived columns like `days_to_response`, `reached_interview`, and `days_since_applied` at load time made every subsequent analysis cleaner and faster to write. Good data design compounds.

Finally, automating the **weekly report generator** shifted my perspective on cron jobs. Scheduling transforms a script from a one-time tool into infrastructure. The same code that runs manually in this notebook would, with one crontab line, run silently every Monday morning forever — no human involvement required. That's the real power of automation in Ubuntu environments.

> *Word count: ~190 words*
